In [9]:
import numpy as np
import pandas as pd

df = pd.read_csv("hantavirus2026.csv")

def clean_hantavirus_optimized(file_path: str) -> pd.DataFrame:
    # 1. Ingestión directa mapeando nulos desde el inicio
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Renombrado masivo y vectorizado de columnas
    df.rename(columns={
        "cruise.crew..y.n.": "is_crew",
        "passenger..y.n.": "is_passenger",
        "Travel_flight": "travel_flight",
        "Gh_ID": "gh_id"
    }, inplace=True)
    
    # 3. Conversión masiva de fechas (Operación Vectorizada)
    date_cols = ["symptom_onset", "outcome_date", "treatment_date", "ship_boarded", "left_ship", "confirmation_date"]
    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")
    
    # 4. Optimización de Edad: Tratamiento de "70s" y numéricos sin bucles
    # Extrae los números. Si el string original contenía una 's', le suma 5 de forma condicional.
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5.0
    
    # 5. Normalización de texto y mapeo binario a Flags de BI (1.0 / 0.0)
    binary_map = {"y": 1.0, "n": 0.0, "yes": 1.0, "no": 0.0}
    df["is_crew"] = df["is_crew"].str.lower().str.strip().map(binary_map)
    df["is_passenger"] = df["is_passenger"].str.lower().str.strip().map(binary_map)
    df["confirmed"] = df["confirmed"].str.lower().str.strip().map(binary_map)
    
    # Estandarizar el resto de categóricos a minúsculas
    categorical_cols = ["status", "outcome", "sex", "nationality", "treatment"]
    df[categorical_cols] = df[categorical_cols].astype(str).apply(lambda x: x.str.lower().str.strip()).replace("nan", np.nan)

    # 6. Corrección de Typos vectorizada con reemplazo directo por diccionario
    typos_dict = {
        "diarreah": "diarrhea", 
        "diarrhiea": "diarrhea", 
        "gastrointesinal": "gastrointestinal",
        "flu like symptoms": "flu-like symptoms"
    }
    df["symptoms"] = df["symptoms"].astype(str).str.lower().str.strip()
    df["symptoms"].replace(typos_dict, regex=True, inplace=True)
    df["symptoms"].replace("nan", "unspecified", inplace=True)

    # 7. Desglose Atómico de Destinos (1FN) usando vectorización de strings
    # Limpia paréntesis, corrige Schiphol y toma el destino final después del 'then'
    travel_clean = df["travel_to"].astype(str).str.replace(r"\(.*?\)", "", regex=True).str.replace("Schipol", "Schiphol")
    final_destination = travel_clean.str.split("then").str[-1].str.strip()
    
    # Divide en Ciudad y País si existe la coma
    df["destination_city"] = final_destination.str.split(",").str[0].str.strip().str.title()
    df["destination_country"] = final_destination.str.split(",").str[1].str.strip().str.title()
    
    # Caso donde no había coma (ej. solo "Netherlands"): El valor queda en el país y limpiamos la ciudad
    only_country_mask = final_destination.str.contains(",") == False
    df.loc[only_country_mask, "destination_country"] = df.loc[only_country_mask, "destination_city"]
    df.loc[only_country_mask, "destination_city"] = np.nan
    
    # Eliminar columna sucia residual y formatear IDs
    df.drop(columns=["travel_to"], inplace=True)
    df["gh_id"] = df["gh_id"].astype(int)
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").astype("Int64")
    
    return df

# Ejecución rápida
if __name__ == "__main__":
    # df_limpio = clean_hantavirus_optimized("hantavirus2026.csv")
    print("Pipeline ultra-optimizado listo para ejecución masiva.")

Pipeline ultra-optimizado listo para ejecución masiva.


In [13]:
df.sample(20)

,Gh_ID,linked_id,status,linked_information,symptom_onset,outcome,outcome_date,treatment,treatment_date,cruise.crew..y.n.,...,Travel_flight,context,ship_boarded,left_ship,left_location,confirmed,confirmation_date,nationality,accession_id,sources
13,14,2.0,tested,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spanish,NaN,https://www.reuters.com/business/healthcare-ph...
19,20,NaN,confirmed,evacuated,2026-05-14,NaN,NaN,quarantine,NaN,n,...,evacuation flight,from Yukon Terriroty. Isolating on Vancouver I...,2026-04-01,2026-05-10,Tenerife,y,2026-05-15,canadian,NaN,https://www.bbc.com/news/articles/c2k2p057ezno...
8,9,4.0,negative,NaN,NaN,NaN,NaN,hospitalised,2026-05-06,n,...,Medicalized aircraft 1,NaN,2026-04-01,2026-05-06,Cape Verde,NaN,NaN,german,NaN,https://www.bbc.com/news/live/cy592qeq071t; ht...
3,4,NaN,confirmed,NaN,2026-04-28,died,2026-05-02,NaN,NaN,n,...,NaN,NaN,2026-04-01,NaN,NaN,NaN,NaN,german,NaN,https://www.bbc.com/news/articles/c5y093d5n9ko
7,8,NaN,confirmed,NaN,2026-04-27,NaN,NaN,hospitalised,2026-05-07,y,...,Medicalized aircraft 2,NaN,2026-04-01,2026-05-06,Cape Verde,y,2026-05-06,dutch,PP_006W6RC.2,https://www.bbc.com/news/live/cy592qeq071t; ht...
10,11,2.0,negative,on flight to johannsburg with women who later ...,NaN,NaN,NaN,hospitalised,NaN,n,...,NaN,"arrived in Singapore May 6, 2026",2026-04-01,2026-04-24,St. Helena,NaN,NaN,singaporean permanent resident,NaN,https://www.straitstimes.com/singapore/2-singa...
12,13,2.0,monitored,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,french,NaN,https://www.france24.com/en/health/20260506-li...
11,12,NaN,probable,NaN,2026-04-28,NaN,NaN,NaN,NaN,n,...,NaN,NaN,2026-04-01,2026-04-14,Tristan de Cunha,NaN,NaN,british,NaN,https://ukhsa-newsroom.prgloo.com/news/uk-gove...
2,3,NaN,confirmed,NaN,2026-04-24,NaN,NaN,intensive care,2026-04-29,n,...,NaN,NaN,2026-04-01,2026-04-27,Ascension,y,2026-05-02,british,PP_00WANE.2,https://www.science.org/content/article/cruise...
4,5,NaN,confirmed,NaN,2026-05-01,NaN,NaN,intensive care,NaN,n,...,NaN,NaN,2026-04-01,2026-04-24,St. Helena,y,2026-05-06,swiss,PP_006WBLH.1,https://www.swissinfo.ch/eng/various/man-with-...


In [15]:
def impute_and_clean_dataset(file_path: str) -> pd.DataFrame:
    # 1. Leer el dataset convirtiendo vacíos y "NA" en objetos NaN reales
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Renombrar columnas clave para estandarizar
    df.rename(columns={
        "cruise.crew..y.n.": "is_crew",
        "passenger..y.n.": "is_passenger",
        "Travel_flight": "travel_flight",
        "Gh_ID": "gh_id"
    }, inplace=True)
    
    # 3. Preprocesamiento rápido de la Edad (Tratamiento del texto "70s" a numérico)
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5.0  # Ajustar el rango "70s" a 75
    
    # Convertir otras columnas numéricas/flags binarias
    binary_map = {"y": 1.0, "n": 0.0, "yes": 1.0, "no": 0.0}
    for col in ["is_crew", "is_passenger", "confirmed"]:
        df[col] = df[col].astype(str).str.lower().str.strip().map(binary_map)

    # =============================================================================
    # ESTRATEGIA DE IMPUTACIÓN DE VALORES NaN
    # =============================================================================
    
    # --- A. Columnas Numéricas: Imputación con la MEDIA ---
    # Identificamos columnas numéricas (excluyendo el 'gh_id' por ser el identificador único)
    numeric_cols = ["age", "is_crew", "is_passenger", "confirmed"]
    
    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            media_col = df[col].mean()
            # Si la columna es un Flag binario (0 o 1), redondeamos la media para mantener la coherencia
            if col in ["is_crew", "is_passenger", "confirmed"]:
                media_col = round(media_col)
            df[col].fillna(media_col, inplace=True)
            print(f"-> Columna numérica '{col}': NaN reemplazados con la media ({media_col:.2f})")

    # --- B. Columnas de Texto / Categóricas: Imputación con la MODA ---
    # Columnas de texto relevantes para el análisis
    text_cols = ["status", "outcome", "sex", "nationality", "symptoms", "travel_from", "left_location"]
    
    for col in text_cols:
        # Aseguramos formato limpio de texto en minúsculas y removemos nan en string residuales
        df[col] = df[col].astype(str).str.lower().str.strip().replace("nan", np.nan)
        
        if df[col].isnull().sum() > 0:
            # Calculamos la moda (el valor más común). [0] toma el primer resultado si hay empate.
            moda_col = df[col].mode()[0] if not df[col].mode().empty else "unspecified"
            df[col].fillna(moda_col, inplace=True)
            print(f"-> Columna de texto '{col}': NaN reemplazados con la moda ('{moda_col}')")

    # --- C. Columnas de Fechas: Tratamiento Especial ---
    # En fechas no se puede aplicar una media aritmética tradicional de manera directa sin romper la lógica temporal. 
    # Lo óptimo en salud es rellenar con una "Fecha de Control/Desconocida" o mantenerlas si representan hitos no ocurridos.
    date_cols = ["symptom_onset", "outcome_date", "treatment_date", "ship_boarded", "left_ship", "confirmation_date"]
    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")
    # Opcional: Rellenar fechas faltantes con una fecha por defecto si tu modelo requiere obligatoriamente datos
    # df[date_cols] = df[date_cols].fillna(pd.Timestamp('2026-01-01')) 

    # --- D. Limpieza final de IDs ---
    df["gh_id"] = df["gh_id"].astype(int)
    # Para el ID vinculado, si es NaN, le asignamos 0 para indicar que no tiene un caso pariente
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)

    return df

# =============================================================================
# EJECUCIÓN DEL SCRIPT
# =============================================================================
if __name__ == "__main__":
    # Reemplaza con la ruta de tu archivo
    PATH_DATASET = "hantavirus2026.csv" 
    
    df_limpio_total = impute_and_clean_dataset(PATH_DATASET)
    
    print("\n### MUESTRA DEL DATASET COMPLETAMENTE SANITIZADO ###")
    print(df_limpio_total[["gh_id", "age", "is_passenger", "status", "sex", "nationality"]].head(10))

-> Columna numérica 'age': NaN reemplazados con la media (63.50)
-> Columna numérica 'confirmed': NaN reemplazados con la media (1.00)
-> Columna de texto 'outcome': NaN reemplazados con la moda ('died')
-> Columna de texto 'sex': NaN reemplazados con la moda ('male')
-> Columna de texto 'nationality': NaN reemplazados con la moda ('british')
-> Columna de texto 'travel_from': NaN reemplazados con la moda ('canary islands')
-> Columna de texto 'left_location': NaN reemplazados con la moda ('st. helena')

### MUESTRA DEL DATASET COMPLETAMENTE SANITIZADO ###
   gh_id   age  is_passenger     status     sex  nationality
0      1  70.0           1.0   probable    male        dutch
1      2  69.0           1.0  confirmed  female        dutch
2      3  63.5           1.0  confirmed    male      british
3      4  63.5           1.0  confirmed  female       german
4      5  63.5           1.0  confirmed    male        swiss
5      6  63.5           0.0   negative  female      british
6      7  

In [16]:
def clean_hantavirus_integer_age(file_path: str) -> pd.DataFrame:
    # 1. Ingestión de datos inicial
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Preprocesamiento de la edad (Aislar números y resolver el rango "70s")
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    
    # Extraemos el número base
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    # Si tenía una 's' (como 70s), le asignamos el punto medio de la década (75)
    df.loc[has_s, "age"] += 5
    
    # =============================================================================
    # SOLUCIÓN CRÍTICA: Imputación con la Media y Conversión Estricta a Entero
    # =============================================================================
    # Calculamos la media de la edad (nos da un decimal, ej: 58.33)
    media_edad = df["age"].mean()
    
    # Rellenamos los NaN con la media redondeada a un número entero lógico (ej: 58)
    df["age"].fillna(round(media_edad), inplace=True)
    
    # Forzamos el tipo de dato a Entero Corporativo 'Int64' (Soporta nulos si aparecieran en el futuro)
    df["age"] = df["age"].astype("Int64")
    
    # =============================================================================
    # Resto del pipeline de estandarización rápido
    # =============================================================================
    df.rename(columns={"cruise.crew..y.n.": "is_crew", "passenger..y.n.": "is_passenger", "Gh_ID": "gh_id"}, inplace=True)
    
    binary_map = {"y": 1, "n": 0}
    df["is_crew"] = df["is_crew"].str.lower().str.strip().map(binary_map).fillna(0).astype(int)
    df["is_passenger"] = df["is_passenger"].str.lower().str.strip().map(binary_map).fillna(0).astype(int)
    
    df["gh_id"] = df["gh_id"].astype(int)
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)
    
    return df

# Ejecución y verificación
if __name__ == "__main__":
    PATH_DATASET = "hantavirus2026.csv"
    df_final = clean_hantavirus_integer_age(PATH_DATASET)
    
    print("### VERIFICACIÓN DEL TIPO DE DATO DE EDAD ###")
    print(df_final[["gh_id", "age"]].head(10).to_string(index=False))
    print(f"\nTipo de dato de la columna 'age': {df_final['age'].dtype}")

### VERIFICACIÓN DEL TIPO DE DATO DE EDAD ###
 gh_id  age
     1   70
     2   69
     3   64
     4   64
     5   64
     6   64
     7   56
     8   41
     9   65
    10   67

Tipo de dato de la columna 'age': Int64


In [17]:
def clean_hantavirus_confirmed_text(file_path: str) -> pd.DataFrame:
    # 1. Ingestión del dataset y limpieza de columnas iniciales
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    df.rename(columns={"cruise.crew..y.n.": "is_crew", "passenger..y.n.": "is_passenger", "Gh_ID": "gh_id"}, inplace=True)
    
    # 2. Estandarizar la columna de estatus clínico a minúsculas
    df["status"] = df["status"].astype(str).str.lower().str.strip()
    
    # =============================================================================
    # TRANSFORMAR LA COLUMNA "CONFIRMED" A TEXTO SEGURO ("yes" / "no")
    # =============================================================================
    
    # Primero limpiamos cualquier entrada residual en la columna 'confirmed' original
    df["confirmed"] = df["confirmed"].astype(str).str.lower().str.strip()
    
    # Regla de negocio lógica avanzada: 
    # Es un caso contagiado ("yes") si la columna 'confirmed' ya dice 'y'/'yes', 
    # O si el estatus general ('status') está catalogado como 'confirmed' o 'probable'.
    is_positive = (df["confirmed"].isin(["y", "yes"])) | (df["status"].isin(["confirmed", "probable"]))
    
    # Aplicamos las etiquetas finales en texto de forma vectorizada
    df["confirmed"] = np.where(is_positive, "yes", "no")
    
    # =============================================================================
    # Formateo complementario e imputación de Edad (Entero puro)
    # =============================================================================
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5
    
    # Imputar la edad con la media entera
    media_edad = df["age"].mean()
    df["age"].fillna(round(media_edad), inplace=True)
    df["age"] = df["age"].astype("Int64")
    
    # Limpieza de identificadores
    df["gh_id"] = df["gh_id"].astype(int)
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)
    
    return df

# Ejecución y prueba de calidad
if __name__ == "__main__":
    PATH_DATASET = "hantavirus2026.csv"
    df_final = clean_hantavirus_confirmed_text(PATH_DATASET)
    
    print("### CONTROL DE CALIDAD: COLUMNA 'CONFIRMED' EN TEXTO ###")
    columns_to_verify = ["gh_id", "status", "age", "confirmed"]
    print(df_final[columns_to_verify].head(10).to_string(index=False))

### CONTROL DE CALIDAD: COLUMNA 'CONFIRMED' EN TEXTO ###
 gh_id    status  age confirmed
     1  probable   70       yes
     2 confirmed   69       yes
     3 confirmed   64       yes
     4 confirmed   64       yes
     5 confirmed   64       yes
     6  negative   64        no
     7 confirmed   56       yes
     8 confirmed   41       yes
     9  negative   65        no
    10  negative   67        no


In [18]:
df.sample(20)

,Gh_ID,linked_id,status,linked_information,symptom_onset,outcome,outcome_date,treatment,treatment_date,cruise.crew..y.n.,...,Travel_flight,context,ship_boarded,left_ship,left_location,confirmed,confirmation_date,nationality,accession_id,sources
16,17,NaN,monitored,evacuated,2026-05-10,NaN,NaN,quarantine,2026-05-10,n,...,evacuation flight,evacuated on 2026-05-10,2026-04-01,2026-05-10,Tenerife,NaN,NaN,american,NaN,https://apnews.com/article/hantavirus-outbreak...
10,11,2.0,negative,on flight to johannsburg with women who later ...,NaN,NaN,NaN,hospitalised,NaN,n,...,NaN,"arrived in Singapore May 6, 2026",2026-04-01,2026-04-24,St. Helena,NaN,NaN,singaporean permanent resident,NaN,https://www.straitstimes.com/singapore/2-singa...
19,20,NaN,confirmed,evacuated,2026-05-14,NaN,NaN,quarantine,NaN,n,...,evacuation flight,from Yukon Terriroty. Isolating on Vancouver I...,2026-04-01,2026-05-10,Tenerife,y,2026-05-15,canadian,NaN,https://www.bbc.com/news/articles/c2k2p057ezno...
9,10,2.0,negative,on flight to johannsburg with women who later ...,NaN,NaN,NaN,hospitalised,NaN,n,...,NaN,"arrived in Singapore May 2, 2026",2026-04-01,2026-04-24,St. Helena,NaN,NaN,singaporean,NaN,https://www.straitstimes.com/singapore/2-singa...
12,13,2.0,monitored,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,french,NaN,https://www.france24.com/en/health/20260506-li...
13,14,2.0,tested,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spanish,NaN,https://www.reuters.com/business/healthcare-ph...
4,5,NaN,confirmed,NaN,2026-05-01,NaN,NaN,intensive care,NaN,n,...,NaN,NaN,2026-04-01,2026-04-24,St. Helena,y,2026-05-06,swiss,PP_006WBLH.1,https://www.swissinfo.ch/eng/various/man-with-...
18,19,NaN,tested,"travelled from Argentina on April 30, 2026",NaN,NaN,NaN,hospitalised,2026-05-12,n,...,commercial flight,flew from Argentina to Rome,NaN,NaN,NaN,NaN,NaN,argentinian,NaN,https://www.ansa.it/sito/notizie/mondo/2026/05...
8,9,4.0,negative,NaN,NaN,NaN,NaN,hospitalised,2026-05-06,n,...,Medicalized aircraft 1,NaN,2026-04-01,2026-05-06,Cape Verde,NaN,NaN,german,NaN,https://www.bbc.com/news/live/cy592qeq071t; ht...
14,15,NaN,confirmed,evacuated,2026-05-10,NaN,NaN,hospitalised,2026-05-10,n,...,evacuation flight,evacuated on 2026-05-10,2026-04-01,2026-05-10,Tenerife,y,2026-05-11,french,NaN,https://www.ft.com/content/6793c1a0-9b84-43ba-...


In [19]:
def clean_hantavirus_ship_boarded(file_path: str) -> pd.DataFrame:
    # 1. Ingestión del dataset mapeando valores nulos iniciales
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Renombrado de columnas (Incluyendo la corrección semántica a 'ship_boarded' si fuera necesario)
    df.rename(columns={
        "cruise.crew..y.n.": "is_crew", 
        "passenger..y.n.": "is_passenger", 
        "Gh_ID": "gh_id",
        "ship_boarded": "ship_boarded"  # Validamos que mantenga su nombre estándar
    }, inplace=True)
    
    # =============================================================================
    # ESTABLECER FECHA DE ABORDAJE ÚNICA Y CONSTANTE (Punto de Partida)
    # =============================================================================
    # Forzamos a que TODOS los registros tengan exactamente la fecha de inicio del crucero
    df["ship_boarded"] = pd.to_datetime("2026-04-01")
    
    # =============================================================================
    # Pipeline de calidad complementario (Pasos anteriores unificados)
    # =============================================================================
    
    # A. Columna 'confirmed' a Texto Seguro ("yes" / "no") basado en validación cruzada
    df["status"] = df["status"].astype(str).str.lower().str.strip()
    df["confirmed"] = df["confirmed"].astype(str).str.lower().str.strip()
    is_positive = (df["confirmed"].isin(["y", "yes"])) | (df["status"].isin(["confirmed", "probable"]))
    df["confirmed"] = np.where(is_positive, "yes", "no")
    
    # B. Edad a Entero Puro (Int64) con imputación de media redondeada
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5
    media_edad = df["age"].mean()
    df["age"].fillna(round(media_edad), inplace=True)
    df["age"] = df["age"].astype("Int64")
    
    # C. Limpieza e indexación de identificadores de paciente
    df["gh_id"] = df["gh_id"].astype(int)
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)
    
    # D. Estandarizar resto de fechas del pipeline para cálculos posteriores
    other_date_cols = ["symptom_onset", "outcome_date", "treatment_date", "left_ship", "confirmation_date"]
    df[other_date_cols] = df[other_date_cols].apply(pd.to_datetime, errors="coerce")
    
    return df

# Ejecución y auditoría de la línea de tiempo
if __name__ == "__main__":
    PATH_DATASET = "hantavirus2026.csv"
    df_final = clean_hantavirus_ship_boarded(PATH_DATASET)
    
    print("### AUDITORÍA: CONSTANTE DE PUNTO DE PARTIDO (SHIP_BOARDED) ###")
    columns_to_check = ["gh_id", "ship_boarded", "symptom_onset", "confirmed"]
    print(df_final[columns_to_check].head(10).to_string(index=False))
    
    # Verificación de que no existan desviaciones ni nulos en la columna fija
    print(f"\nValores únicos en 'ship_boarded': {df_final['ship_boarded'].dt.strftime('%Y-%m-%d').unique()}")
    print(f"Total registros verificados: {len(df_final)}")

### AUDITORÍA: CONSTANTE DE PUNTO DE PARTIDO (SHIP_BOARDED) ###
 gh_id ship_boarded symptom_onset confirmed
     1   2026-04-01    2026-04-06       yes
     2   2026-04-01    2026-04-24       yes
     3   2026-04-01    2026-04-24       yes
     4   2026-04-01    2026-04-28       yes
     5   2026-04-01    2026-05-01       yes
     6   2026-04-01           NaT        no
     7   2026-04-01    2026-04-30       yes
     8   2026-04-01    2026-04-27       yes
     9   2026-04-01           NaT        no
    10   2026-04-01           NaT        no

Valores únicos en 'ship_boarded': ['2026-04-01']
Total registros verificados: 20


In [20]:
def clean_hantavirus_incubation_period(file_path: str) -> pd.DataFrame:
    # 1. Ingestión del dataset mapeando nulos iniciales
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Renombrado de columnas esenciales
    df.rename(columns={
        "cruise.crew..y.n.": "is_crew", 
        "passenger..y.n.": "is_passenger", 
        "Gh_ID": "gh_id"
    }, inplace=True)
    
    # 3. Establecer el punto de partida fijo (datetime corporativo)
    df["ship_boarded"] = pd.to_datetime("2026-04-01")
    
    # =============================================================================
    # REGLA DE NEGOCIO: CALCULAR SYMPTOM_ONSET (18 días después del abordaje)
    # =============================================================================
    # Usamos pd.Timedelta para sumar los días directamente de forma vectorizada en memoria
    df["symptom_onset"] = df["ship_boarded"] + pd.Timedelta(days=18)
    
    # =============================================================================
    # Pipeline complementario de calidad (Unificado)
    # =============================================================================
    
    # Columna 'confirmed' a Texto Seguro ("yes" / "no")
    df["status"] = df["status"].astype(str).str.lower().str.strip()
    df["confirmed"] = df["confirmed"].astype(str).str.lower().str.strip()
    is_positive = (df["confirmed"].isin(["y", "yes"])) | (df["status"].isin(["confirmed", "probable"]))
    df["confirmed"] = np.where(is_positive, "yes", "no")
    
    # Edad a Entero Puro (Int64) con imputación de media redondeada
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5
    media_edad = df["age"].mean()
    df["age"].fillna(round(media_edad), inplace=True)
    df["age"] = df["age"].astype("Int64")
    
    # Formateo de IDs
    df["gh_id"] = df["gh_id"].astype(int)
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)
    
    # Estandarizar el resto de columnas tipo fecha
    other_date_cols = ["outcome_date", "treatment_date", "left_ship", "confirmation_date"]
    df[other_date_cols] = df[other_date_cols].apply(pd.to_datetime, errors="coerce")
    
    return df

# Ejecución y auditoría de la línea de tiempo recalculada
if __name__ == "__main__":
    PATH_DATASET = "hantavirus2026.csv"
    df_final = clean_hantavirus_incubation_period(PATH_DATASET)
    
    print("### AUDITORÍA DE LÍNEA DE TIEMPO CON PERÍODO DE INCUBACIÓN MEDIA ###")
    columns_to_check = ["gh_id", "ship_boarded", "symptom_onset", "confirmed", "age"]
    print(df_final[columns_to_check].head(10).to_string(index=False))
    
    # Verificación de consistencia del cálculo
    print(f"\nFecha única de inicio de síntomas: {df_final['symptom_onset'].dt.strftime('%Y-%m-%d').unique()}")

### AUDITORÍA DE LÍNEA DE TIEMPO CON PERÍODO DE INCUBACIÓN MEDIA ###
 gh_id ship_boarded symptom_onset confirmed  age
     1   2026-04-01    2026-04-19       yes   70
     2   2026-04-01    2026-04-19       yes   69
     3   2026-04-01    2026-04-19       yes   64
     4   2026-04-01    2026-04-19       yes   64
     5   2026-04-01    2026-04-19       yes   64
     6   2026-04-01    2026-04-19        no   64
     7   2026-04-01    2026-04-19       yes   56
     8   2026-04-01    2026-04-19       yes   41
     9   2026-04-01    2026-04-19        no   65
    10   2026-04-01    2026-04-19        no   67

Fecha única de inicio de síntomas: ['2026-04-19']


In [21]:
df.sample(20)

,Gh_ID,linked_id,status,linked_information,symptom_onset,outcome,outcome_date,treatment,treatment_date,cruise.crew..y.n.,...,Travel_flight,context,ship_boarded,left_ship,left_location,confirmed,confirmation_date,nationality,accession_id,sources
11,12,NaN,probable,NaN,2026-04-28,NaN,NaN,NaN,NaN,n,...,NaN,NaN,2026-04-01,2026-04-14,Tristan de Cunha,NaN,NaN,british,NaN,https://ukhsa-newsroom.prgloo.com/news/uk-gove...
16,17,NaN,monitored,evacuated,2026-05-10,NaN,NaN,quarantine,2026-05-10,n,...,evacuation flight,evacuated on 2026-05-10,2026-04-01,2026-05-10,Tenerife,NaN,NaN,american,NaN,https://apnews.com/article/hantavirus-outbreak...
13,14,2.0,tested,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,spanish,NaN,https://www.reuters.com/business/healthcare-ph...
15,16,NaN,inconclusive,evacuated,NaN,NaN,NaN,"biocontainment unit, Nebraska",2026-05-10,n,...,evacuation flight,evacuated on 2026-05-10,2026-04-01,2026-05-10,Tenerife,y,2026-05-11,american,NaN,https://apnews.com/article/hantavirus-outbreak...
10,11,2.0,negative,on flight to johannsburg with women who later ...,NaN,NaN,NaN,hospitalised,NaN,n,...,NaN,"arrived in Singapore May 6, 2026",2026-04-01,2026-04-24,St. Helena,NaN,NaN,singaporean permanent resident,NaN,https://www.straitstimes.com/singapore/2-singa...
12,13,2.0,monitored,on flight to johannsburg with women who later ...,NaN,NaN,NaN,monitored,NaN,n,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,french,NaN,https://www.france24.com/en/health/20260506-li...
0,1,2.0,probable,husband ID2,2026-04-06,died,2026-04-11,NaN,NaN,n,...,NaN,NaN,2026-04-01,2026-04-24,St. Helena,NaN,NaN,dutch,NaN,https://www.theguardian.com/world/2026/may/07/...
7,8,NaN,confirmed,NaN,2026-04-27,NaN,NaN,hospitalised,2026-05-07,y,...,Medicalized aircraft 2,NaN,2026-04-01,2026-05-06,Cape Verde,y,2026-05-06,dutch,PP_006W6RC.2,https://www.bbc.com/news/live/cy592qeq071t; ht...
4,5,NaN,confirmed,NaN,2026-05-01,NaN,NaN,intensive care,NaN,n,...,NaN,NaN,2026-04-01,2026-04-24,St. Helena,y,2026-05-06,swiss,PP_006WBLH.1,https://www.swissinfo.ch/eng/various/man-with-...
1,2,1.0,confirmed,wife ID1,2026-04-24,died,2026-04-26,NaN,NaN,n,...,Airlink,"82 passengers, 6 crew, boarded onward flight t...",2026-04-01,2026-04-24,St. Helena,y,2026-05-04,dutch,PP_006WDKH.1,https://www.theguardian.com/world/2026/may/07/...


In [ ]:
def clean_hantavirus_hospitalization_protocol(file_path: str) -> pd.DataFrame:
    # 1. Ingestión del dataset mapeando nulos iniciales
    df = pd.read_csv(file_path, na_values=["NA", "na", ""], keep_default_na=True)
    
    # 2. Renombrado e inicialización de columnas esenciales
    df.rename(columns={
        "cruise.crew..y.n.": "is_crew", 
        "passenger..y.n.": "is_passenger", 
        "Gh_ID": "gh_id"
    }, inplace=True)
    
    # Asegurar que las columnas de análisis estén en minúsculas y limpias de espacios
    df["status"] = df["status"].astype(str).str.lower().str.strip()
    df["treatment"] = df["treatment"].astype(str).str.lower().str.strip().replace("nan", np.nan)
    
    # =============================================================================
    # REGLA DE NEGOCIO: PROTOCOLO DE HOSPITALIZACIÓN AUTOMÁTICA
    # =============================================================================
    # Evaluamos la condición: ¿El estatus es 'confirmed' o 'probable'?
    requires_hospitalization = df["status"].isin(["confirmed", "probable"])
    
    # Si cumple la condición, forzamos 'hospitalised'. 
    # Si no la cumple, conserva su valor original de la columna 'treatment'
    df["treatment"] = np.where(requires_hospitalization, "hospitalised", df["treatment"])
    
    # Para los registros que aún queden como NaN en treatment (ej. casos negativos o testeados),
    # aplicamos por defecto una etiqueta de control neutral
    df["treatment"].fillna("unspecified", inplace=True)

    # =============================================================================
    # Bloque de Calidad Avanzado (Reglas consolidadas anteriormente)
    # =============================================================================
    
    # A. Fechas constantes y cálculo del periodo medio de incubación (18 días)
    df["ship_boarded"] = pd.to_datetime("2026-04-01")
    df["symptom_onset"] = df["ship_boarded"] + pd.Timedelta(days=18)
    
    # B. Columna 'confirmed' a texto limpio ("yes" / "no")
    df["confirmed"] = df["confirmed"].astype(str).str.lower().str.strip()
    is_positive = (df["confirmed"].isin(["y", "yes"])) | (df["status"].isin(["confirmed", "probable"]))
    df["confirmed"] = np.where(is_positive, "yes", "no")
    
    # C. Edad a Entero Puro (Int64) con imputación de media redondeada
    age_clean = df["age"].astype(str).str.lower()
    has_s = age_clean.str.contains("s", na=False)
    df["age"] = pd.to_numeric(age_clean.str.extract(r"(\d+)", expand=False), errors="coerce")
    df.loc[has_s, "age"] += 5
    media_edad = df["age"].mean()
    df["age"].fillna(round(media_edad), inplace=True)
    df["age"] = df["age"].astype("Int64")
    
    # D. Formateo estricto de identificadores numéricos
    df["gh_id"] = df["gh_id"].astype(int)
    df["linked_id"] = pd.to_numeric(df["linked_id"], errors="coerce").fillna(0).astype(int)
    
    # E. Casteo de otras columnas de fechas residuales
    other_date_cols = ["outcome_date", "treatment_date", "left_ship", "confirmation_date"]
    df[other_date_cols] = df[other_date_cols].apply(pd.to_datetime, errors="coerce")
    
    return df

# Ejecución del pipeline y auditoría del conteo
if __name__ == "__main__":
    PATH_DATASET = "hantavirus2026.csv"
    df_final = clean_hantavirus_hospitalization_protocol(PATH_DATASET)
    
    print("### CONTEO DE CASOS POR TIPO DE TRATAMIENTO (POST-REGLA) ###")
    print(df_final["treatment"].value_counts())
    print("\n" + "="*60 + "\n")
    
    print("### AUDITORÍA CRUZADA: STATUS vs TREATMENT ###")
    columns_to_show = ["gh_id", "status", "treatment", "confirmed"]
    # Mostramos los primeros 12 registros para verificar la asignación
    print(df_final[columns_to_show].head(12).to_string(index=False))